In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/train/train_data.csv")

df["date"] = pd.to_datetime(df["date"])

df = (
    df.sort_values(["location_id", "date"])
    .reset_index(drop=True)
)

print("Kích thước dữ liệu:", df.shape)
print("Thời gian:", df["date"].min(), "->", df["date"].max())

df.head()

Kích thước dữ liệu: (18286, 18)
Thời gian: 2001-07-01 00:00:00 -> 2026-07-12 00:00:00


,location_id,location,date,rainfall,temperature,humidity,pressure,wind_speed,river_discharge,flood,target_d1,target_d2,target_d3,rainfall_3d,rainfall_7d,discharge_change_1d,month,season
0,1,Hue,2001-07-01,0.0,28.600000,68.0,1000.400024,24.799999,0.53,0,0.0,0.0,0.0,0.0,0.0,0.00,7,dry_season
1,1,Hue,2001-07-02,0.2,29.100000,70.0,998.700012,20.900000,0.44,0,0.0,0.0,0.0,0.2,0.2,-0.09,7,dry_season
2,1,Hue,2001-07-03,1.8,28.600000,79.0,999.599976,6.900000,0.42,0,0.0,0.0,0.0,2.0,2.0,-0.02,7,dry_season
3,1,Hue,2001-07-04,13.8,27.799999,85.0,999.900024,11.500000,0.44,0,0.0,0.0,0.0,15.8,15.8,0.02,7,dry_season
4,1,Hue,2001-07-05,0.0,28.799999,74.0,997.500000,25.200001,0.41,0,0.0,0.0,0.0,15.6,15.8,-0.03,7,dry_season


In [3]:
from sklearn.model_selection import train_test_split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size =0.2)

NameError: name 'X' is not defined

In [3]:
features = [
    "location_id",
    "rainfall",
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "river_discharge",
    "rainfall_3d",
    "rainfall_7d",
    "discharge_change_1d",
    "month"
]

targets = [
    "target_d1",
    "target_d2",
    "target_d3"
]

In [4]:
TRAIN_END = "2022-01-01"

train_df = df[
    df["date"] < TRAIN_END
]

test_df = df[
    df["date"] >= TRAIN_END
]

print(train_df.shape)
print(test_df.shape)

(14978, 18)
(3308, 18)


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [6]:
models = {
    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
}

# METRICES

In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

In [8]:
def evaluate(
    y_true,
    y_prob,
    threshold
):

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    return {
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),

        "PR-AUC": average_precision_score(
            y_true,
            y_prob
        )
    }

# TRAIN MODEL

In [9]:
results = []

for target in targets:

    current_train = train_df.dropna(
        subset=[target]
    )

    current_test = test_df.dropna(
        subset=[target]
    )

    X_train = current_train[
        features
    ]

    y_train = current_train[
        target
    ]

    X_test = current_test[
        features
    ]

    y_test = current_test[
        target
    ]

    for model_name, model in models.items():

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        y_prob = model.predict_proba(
            X_test
        )[:,1]

        metric = evaluate(
            y_test,
            y_pred,
            y_prob
        )

        metric["Target"] = target
        metric["Model"] = model_name

        results.append(metric)

In [10]:
results_df = pd.DataFrame(results)

results_df

,Precision,Recall,F1,PR-AUC,Target,Model
0,0.845411,0.657895,0.739958,0.611243,target_d1,Logistic Regression
1,0.819413,0.682331,0.744615,0.610230,target_d1,Random Forest
2,0.814433,0.446328,0.576642,0.452487,target_d2,Logistic Regression
3,0.775862,0.508475,0.614334,0.473501,target_d2,Random Forest
4,0.620253,0.277358,0.383312,0.288023,target_d3,Logistic Regression
5,0.631769,0.330189,0.433705,0.316114,target_d3,Random Forest
